In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import torch
import pandas as pd
import numpy as np
from collections import Counter
from sklearn.model_selection import train_test_split
from collections import Counter
import re
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
import joblib

device = torch.device("cpu")
MIXED_PRECISION = False

DATA_PATH = "/content/drive/MyDrive/dataset.csv"

df = pd.read_csv(DATA_PATH)
print("Rows:", len(df))
print("Columns:", list(df.columns))

Rows: 96199
Columns: ['id', 'gender', 'age', 'industry', 'text', 'word_count', 'age_group']


In [ ]:
df = df[["id", "gender", "text"]]

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np

SEED = 42
AUTHOR_COL = "id"

def majority(s):
    return s.mode().iloc[0] if not s.mode().empty else s.iloc[0]

author_meta = (
    df.groupby(AUTHOR_COL, as_index=False)
      .agg({"gender": majority, "text": "size"})
      .rename(columns={"text": "n_posts"})
)

author_meta["stratum"] = author_meta["gender"].astype(str)

def safe_split(ids, y, test_size, seed):
    try:
        a, b = train_test_split(ids, test_size=test_size,
                                random_state=seed, stratify=y)
    except ValueError:
        a, b = train_test_split(ids, test_size=test_size,
                                random_state=seed, stratify=None)
    return a, b

ids = author_meta[AUTHOR_COL]
y   = author_meta["stratum"]

train_authors, temp_authors = safe_split(ids, y, test_size=0.20, seed=SEED)

temp_meta = author_meta[author_meta[AUTHOR_COL].isin(temp_authors)]
va_authors, te_authors = safe_split(
    temp_meta[AUTHOR_COL], temp_meta["stratum"], test_size=0.50, seed=SEED
)

train_df = df[df[AUTHOR_COL].isin(train_authors)].copy()
val_df   = df[df[AUTHOR_COL].isin(va_authors)].copy()
test_df  = df[df[AUTHOR_COL].isin(te_authors)].copy()

print("Authors | train/val/test:", len(train_authors), len(va_authors), len(te_authors))
print("Rows    | train/val/test:", len(train_df), len(val_df), len(test_df))

Authors | train/val/test: 4381 548 548
Rows    | train/val/test: 79177 8639 8383


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, confusion_matrix

X_train, y_train = train_df["text"], train_df["gender"]
X_val, y_val     = val_df["text"], val_df["gender"]
X_test, y_test   = test_df["text"], test_df["gender"]

vectorizer = TfidfVectorizer(max_features=20000, ngram_range=(1,2))
X_train_vec = vectorizer.fit_transform(X_train)
X_val_vec   = vectorizer.transform(X_val)
X_test_vec  = vectorizer.transform(X_test)

nb = MultinomialNB()
nb.fit(X_train_vec, y_train)

y_val_pred = nb.predict(X_val_vec)
print("Validation results:")
print(classification_report(y_val, y_val_pred))
print("Confusion matrix:\n", confusion_matrix(y_val, y_val_pred))

y_test_pred = nb.predict(X_test_vec)
print("\nTest results:")
print(classification_report(y_test, y_test_pred))
print("Confusion matrix:\n", confusion_matrix(y_test, y_test_pred))

Validation results:
              precision    recall  f1-score   support

      female       0.64      0.74      0.68      4105
        male       0.72      0.62      0.67      4534

    accuracy                           0.68      8639
   macro avg       0.68      0.68      0.68      8639
weighted avg       0.68      0.68      0.68      8639

Confusion matrix:
 [[3026 1079]
 [1717 2817]]

Test results:
              precision    recall  f1-score   support

      female       0.64      0.70      0.67      4080
        male       0.69      0.64      0.66      4303

    accuracy                           0.67      8383
   macro avg       0.67      0.67      0.67      8383
weighted avg       0.67      0.67      0.67      8383

Confusion matrix:
 [[2842 1238]
 [1568 2735]]


In [ ]:
# ---------------------------- AGE

In [ ]:
DATA_PATH = "/content/drive/MyDrive/dataset.csv"
df = pd.read_csv(DATA_PATH)

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np

SEED = 42
AUTHOR_COL = "id"

def majority(s):
    return s.mode().iloc[0] if not s.mode().empty else s.iloc[0]

author_meta = (
    df.groupby(AUTHOR_COL, as_index=False)
      .agg({"age_group": majority, "text": "size"})
      .rename(columns={"text": "n_posts"})
)

author_meta["stratum"] = author_meta["age_group"].astype(str)

def safe_split(ids, y, test_size, seed):
    try:
        a, b = train_test_split(ids, test_size=test_size,
                                random_state=seed, stratify=y)
    except ValueError:
        a, b = train_test_split(ids, test_size=test_size,
                                random_state=seed, stratify=None)
    return a, b

ids = author_meta[AUTHOR_COL]
y   = author_meta["stratum"]

train_authors, temp_authors = safe_split(ids, y, test_size=0.20, seed=SEED)

temp_meta = author_meta[author_meta[AUTHOR_COL].isin(temp_authors)]
va_authors, te_authors = safe_split(
    temp_meta[AUTHOR_COL], temp_meta["stratum"], test_size=0.50, seed=SEED
)

train_df = df[df[AUTHOR_COL].isin(train_authors)].copy()
val_df   = df[df[AUTHOR_COL].isin(va_authors)].copy()
test_df  = df[df[AUTHOR_COL].isin(te_authors)].copy()

print("Authors | train/val/test:", len(train_authors), len(va_authors), len(te_authors))
print("Rows    | train/val/test:", len(train_df), len(val_df), len(test_df))

Authors | train/val/test: 4381 548 548
Rows    | train/val/test: 77197 9005 9997


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, confusion_matrix

X_train, y_train = train_df["text"], train_df["age_group"]
X_val, y_val     = val_df["text"], val_df["age_group"]
X_test, y_test   = test_df["text"], test_df["age_group"]

vectorizer = TfidfVectorizer(max_features=20000, ngram_range=(1,2))
X_train_vec = vectorizer.fit_transform(X_train)
X_val_vec   = vectorizer.transform(X_val)
X_test_vec  = vectorizer.transform(X_test)

nb = MultinomialNB()
nb.fit(X_train_vec, y_train)

y_val_pred = nb.predict(X_val_vec)
print("Validation results:")
print(classification_report(y_val, y_val_pred))
print("Confusion matrix:\n", confusion_matrix(y_val, y_val_pred))

y_test_pred = nb.predict(X_test_vec)
print("\nTest results:")
print(classification_report(y_test, y_test_pred))
print("Confusion matrix:\n", confusion_matrix(y_test, y_test_pred))

Validation results:
                precision    recall  f1-score   support

Group1 (13–17)       0.87      0.57      0.69      1464
Group2 (18–29)       0.67      0.84      0.75      5488
Group3 (30–48)       0.33      0.19      0.24      2053

      accuracy                           0.65      9005
     macro avg       0.62      0.54      0.56      9005
  weighted avg       0.63      0.65      0.62      9005

Confusion matrix:
 [[ 839  571   54]
 [ 115 4625  748]
 [   7 1657  389]]

Test results:
                precision    recall  f1-score   support

Group1 (13–17)       0.79      0.42      0.55      2030
Group2 (18–29)       0.65      0.85      0.74      5823
Group3 (30–48)       0.45      0.28      0.34      2144

      accuracy                           0.64      9997
     macro avg       0.63      0.51      0.54      9997
  weighted avg       0.64      0.64      0.61      9997

Confusion matrix:
 [[ 845 1109   76]
 [ 217 4945  661]
 [   6 1545  593]]


In [ ]:
# ------------------- Industry ---------------------------

In [ ]:
DATA_PATH = "/content/drive/MyDrive/dataset.csv"
df = pd.read_csv(DATA_PATH)

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np

SEED = 42
AUTHOR_COL = "id"

def majority(s):
    return s.mode().iloc[0] if not s.mode().empty else s.iloc[0]

author_meta = (
    df.groupby(AUTHOR_COL, as_index=False)
      .agg({"industry": majority, "text": "size"})
      .rename(columns={"text": "n_posts"})
)

author_meta["stratum"] = author_meta["industry"].astype(str)

def safe_split(ids, y, test_size, seed):
    try:
        a, b = train_test_split(ids, test_size=test_size,
                                random_state=seed, stratify=y)
    except ValueError:
        a, b = train_test_split(ids, test_size=test_size,
                                random_state=seed, stratify=None)
    return a, b

ids = author_meta[AUTHOR_COL]
y   = author_meta["stratum"]

train_authors, temp_authors = safe_split(ids, y, test_size=0.20, seed=SEED)

temp_meta = author_meta[author_meta[AUTHOR_COL].isin(temp_authors)]
va_authors, te_authors = safe_split(
    temp_meta[AUTHOR_COL], temp_meta["stratum"], test_size=0.50, seed=SEED
)

train_df = df[df[AUTHOR_COL].isin(train_authors)].copy()
val_df   = df[df[AUTHOR_COL].isin(va_authors)].copy()
test_df  = df[df[AUTHOR_COL].isin(te_authors)].copy()

print("Authors | train/val/test:", len(train_authors), len(va_authors), len(te_authors))
print("Rows    | train/val/test:", len(train_df), len(val_df), len(test_df))

Authors | train/val/test: 4381 548 548
Rows    | train/val/test: 77986 8713 9500


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, confusion_matrix

X_train, y_train = train_df["text"], train_df["industry"]
X_val, y_val     = val_df["text"], val_df["industry"]
X_test, y_test   = test_df["text"], test_df["industry"]

vectorizer = TfidfVectorizer(max_features=20000, ngram_range=(1,2))
X_train_vec = vectorizer.fit_transform(X_train)
X_val_vec   = vectorizer.transform(X_val)
X_test_vec  = vectorizer.transform(X_test)

nb = MultinomialNB()
nb.fit(X_train_vec, y_train)

y_val_pred = nb.predict(X_val_vec)
print("Validation results:")
print(classification_report(y_val, y_val_pred))
print("Confusion matrix:\n", confusion_matrix(y_val, y_val_pred))

y_test_pred = nb.predict(X_test_vec)
print("\nTest results:")
print(classification_report(y_test, y_test_pred))
print("Confusion matrix:\n", confusion_matrix(y_test, y_test_pred))

Validation results:
                             precision    recall  f1-score   support

                       Arts       0.19      0.39      0.25       938
      Business & Consulting       0.05      0.00      0.01       631
       Communications-Media       0.11      0.11      0.11       456
   Creative Media & Culture       0.23      0.23      0.23       774
                  Education       0.18      0.15      0.16      1057
         Finance & Property       0.00      0.00      0.00       219
         Industrial & Misc.       0.06      0.00      0.00       666
                   Internet       1.00      0.01      0.01       734
 Law & Specialized Services       0.12      0.21      0.16       406
                 Non-Profit       0.00      0.00      0.00       441
Public Service & Governance       0.00      0.00      0.00       340
        Science & Technical       0.34      0.05      0.09       626
                    Student       0.28      0.69      0.39       856
             

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/m

                             precision    recall  f1-score   support

                       Arts       0.15      0.31      0.21      1063
      Business & Consulting       0.11      0.01      0.01       461
       Communications-Media       0.13      0.07      0.09       743
   Creative Media & Culture       0.12      0.16      0.14       511
                  Education       0.22      0.16      0.18      1327
         Finance & Property       0.00      0.00      0.00       106
         Industrial & Misc.       0.12      0.00      0.01       575
                   Internet       0.00      0.00      0.00       602
 Law & Specialized Services       0.08      0.18      0.11       309
                 Non-Profit       0.00      0.00      0.00       376
Public Service & Governance       0.00      0.00      0.00       401
        Science & Technical       0.17      0.01      0.03       943
                    Student       0.30      0.77      0.43      1088
                 Technology      